# Taming Structured Data Foundation Models with AutoML: Forecasting with AutoGluon

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Innixma/kdd2026_tutorial_materials/blob/main/notebooks/kdd-tutorial-timeseries.ipynb)

*KDD 2026 Hands-On Tutorial*

Time series forecasting drives decisions everywhere: how much inventory to stock, how many servers to
provision, how much energy a grid will need tomorrow. Until recently, building an accurate forecaster
meant hand-picking models, tuning hyperparameters, and writing a lot of glue code. Two shifts have
changed this:

1. **AutoML** — frameworks that automatically train, tune, and ensemble many models behind a single API.
2. **Time series foundation models** — models pretrained on billions of observations that produce
   accurate forecasts *out of the box*, with zero training on your data.

In this tutorial we use [**AutoGluon-TimeSeries**](https://auto.gluon.ai/stable/tutorials/timeseries/index.html)
(AG-TS) to climb the full ladder of forecasting approaches, so you can see exactly what each rung buys you:

| Section | Approach | Training cost |
|---|---|---|
| 1 | Simple baselines (Naive, Seasonal Naive, Average) | ~instant |
| 2 | Classical + ML models (ETS, LightGBM, PatchTST) + ensemble | minutes |
| 3 | **Foundation models** (Chronos-2, Toto-2), zero-shot | seconds |
| 4 | Foundation models **with covariates** + **fine-tuning** | seconds–minutes |

By the end you'll be able to go from a raw file to a state-of-the-art probabilistic forecast in a few
lines of code.

## Setup

AG-TS bundles statistical models, tree-based models (LightGBM), deep learning models, and pretrained
foundation models behind one interface. A GPU is recommended for the foundation-model sections but not
required — the small Chronos-2 and Toto-2 checkpoints run comfortably on CPU.

In [ ]:
# We use uv for faster installation
!pip install uv
!uv pip install -q autogluon.timeseries --system
!uv pip uninstall -q torchaudio torchvision torchtext --system  # fix incompatible package versions on Colab

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt

from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

## 1. Time series forecasting basics

### Loading data

AG-TS uses just two classes:

- **`TimeSeriesDataFrame`** stores a *panel* — a collection of many time series.
- **`TimeSeriesPredictor`** fits, tunes, selects, and ensembles models, then generates forecasts.

We'll start with a subset of the [Australian Electricity Demand dataset](https://zenodo.org/records/4659727):
**half-hourly electricity demand for 5 Australian regions**. It's small and easy to reason about —
demand has a strong daily cycle, so we can *see* whether a model is doing something sensible.

AG-TS expects data in *long format*: one row per (time series ID, timestamp, value).

In [ ]:
data = TimeSeriesDataFrame.from_path("https://autogluon.s3.amazonaws.com/datasets/timeseries/australian_electricity_subset/test.csv")
data.head()

Each row is identified by a series ID (`item_id`) and a `timestamp`, with the observed value in the
`target` column. We refer to each individual time series in the panel as an *item* — here, one region.

Let's confirm how many series we have and at what frequency they're sampled.

In [ ]:
print(f"{data.num_items} time series, frequency = '{data.freq}'")
data.head()

Since `TimeSeriesDataFrame` inherits from `pandas.DataFrame`, we can slice a single item and plot it
with plain pandas to eyeball the raw data before modeling.

In [ ]:
item_id = data.item_ids[0]
data.loc[item_id]["target"].plot(figsize=(12, 3), title=f"Electricity demand — region '{item_id}'")
plt.show()

### The forecasting task

The goal of forecasting is to predict future values given the history. We must choose the
**prediction length** (a.k.a. forecast horizon) — how many steps into the future we want to predict.
The data is half-hourly (48 steps per day), so `prediction_length=48` forecasts one full day ahead.

To evaluate honestly, we hold out data at the end of every series as a test set. We reserve
`num_test_windows=3` days worth of data — enough to *backtest* over several forecast windows later.
`train_data` contains only the history; `test_data` contains history + the future we'll score against.

In [ ]:
prediction_length = 48
num_test_windows = 3
train_data, test_data = data.train_test_split(num_test_windows * prediction_length)

### Start simple: three baselines

Before reaching for anything fancy, always fit trivial baselines — they set the bar every real model
must clear, and are sometimes surprisingly hard to beat.

- **`Average`** — predict the mean of the history.
- **`Naive`** — predict the last observed value.
- **`SeasonalNaive`** — repeat the value from one season ago (one day ago). AG-TS infers the seasonal
  period automatically from the data frequency.

We pass these explicitly via `hyperparameters` and disable the ensemble so we can inspect each model
on its own.

In [ ]:
baseline_predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    target="target",
    eval_metric="MASE",
).fit(
    train_data,
    hyperparameters={"Average": {}, "Naive": {}, "SeasonalNaive": {}},
    enable_ensemble=False,
)

Let's visualize each baseline's forecast for a single region. Note that AG-TS always produces a
**probabilistic** forecast — a mean plus quantiles — not just a single number.

In [ ]:
for model in baseline_predictor.model_names():
    predictions = baseline_predictor.predict(train_data, model=model)
    baseline_predictor.plot(
        test_data,
        predictions,
        item_ids=[item_id],
        max_history_length=200,
    )
    plt.suptitle(f"{model} forecast")
    plt.tight_layout()
    plt.show()

The forecasts look plausible — `SeasonalNaive` echoes yesterday's daily shape, and even `Naive` (a
flat line at the last value) isn't unreasonable over a single day. But eyeballing one series can't tell
us which model is actually best, and it doesn't scale. We need to *quantify* accuracy across all series.

### Choosing a metric: mind the scale

A subtle but important point: our series live on **very different scales** — some regions consume an
order of magnitude more electricity than others. Let's look at the average demand per series.

In [ ]:
avg_per_series = data.groupby(level="item_id")["target"].mean()
avg_per_series.plot.bar(figsize=(7, 3), title="Average demand per series")
plt.ylabel("Mean target value")
plt.tight_layout()
plt.show()

Why does this matter? A scale-dependent metric like **MAE** (Mean Absolute Error) is measured in the
target's raw units, so the few high-demand regions would dominate the average error — a model could
look great overall while forecasting the smaller regions poorly.

Instead we use [**MASE**](https://en.wikipedia.org/wiki/Mean_absolute_scaled_error) (Mean Absolute
Scaled Error). MASE divides each series' error by the error of a naive seasonal forecast *on that same
series*, making the score **scale-free**: every series contributes comparably, regardless of its
magnitude. A MASE below 1 means we beat the seasonal-naive baseline.

### Quantifying accuracy with the leaderboard

`leaderboard()` scores every model on the held-out `test_data`.

> **Note:** AG-TS always reports metrics in **higher-is-better** form. Error metrics that are naturally
> lower-is-better (MASE, MAE, RMSE, …) are multiplied by `-1` in the logs and leaderboard, so a score
> of `-0.5` is better than `-2.0` — closer to zero is best.

In [ ]:
baseline_predictor.leaderboard(test_data)

Now the ranking is unambiguous: `SeasonalNaive` clearly wins among the baselines. This is the number to beat.

## 2. Machine learning models for forecasting

Baselines are a floor. Now let's bring in models that actually *learn* patterns:

- **`ETS`** — exponential smoothing, a classic statistical model for trend + seasonality.
- **`RecursiveTabular`** — turns forecasting into a tabular regression problem and fits LightGBM.
- **`PatchTST`** — a transformer-based deep learning model for time series.

This time we also let AG-TS build a **weighted ensemble** on top (the default — we simply don't set
`enable_ensemble=False`), and cap total training time with `time_limit`.

In [ ]:
ml_predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    target="target",
    eval_metric="MASE",
).fit(
    train_data,
    hyperparameters={
        "SeasonalNaive": {},
        "ETS": {},
        "RecursiveTabular": {},
        "PatchTST": {},
    },
    time_limit=120,
)

In [ ]:
ml_predictor.leaderboard(test_data)

The ML models — and especially the `WeightedEnsemble` combining them — comfortably beat every
baseline. AutoGluon picked and weighted the ensemble automatically; you didn't tune a single knob.

### Are the uncertainty estimates any good?

MASE only measures the point (mean) forecast. But we also got quantiles for free. To check how well
*calibrated* those quantiles are, we evaluate with a **quantile loss** metric. `MQL` (Mean Quantile
Loss) and `WQL` (Weighted Quantile Loss) measure the accuracy of the full probabilistic forecast.

In [ ]:
ml_predictor.evaluate(test_data, metrics=["MASE", "WQL", "MQL"])

### Visualizing the forecast across multiple windows

The real value of a probabilistic forecast is the *range* of outcomes. Because we reserved
`num_test_windows` days of test data, we can **backtest** — generate forecasts for several consecutive
windows — and plot them together with the 10%–90% prediction interval. The shaded band shows where the
true value is expected to land 80% of the time; dashed lines mark each window's cutoff.

In [ ]:
predictions_per_window = ml_predictor.backtest_predictions(test_data, num_val_windows=num_test_windows)

item_ids = test_data.item_ids[:2].tolist()
all_predictions = pd.concat(predictions_per_window)
ml_predictor.plot(test_data, all_predictions, item_ids=item_ids, max_history_length=300)

# Mark the cutoff dates with dashed vertical lines
for cutoff in range(-num_test_windows * prediction_length, 0, prediction_length):
    for i, ax in enumerate(plt.gcf().axes):
        cutoff_timestamp = test_data.loc[item_ids[i]].index[cutoff]
        ax.axvline(cutoff_timestamp, color="gray", linestyle="--")
plt.show()

## 3. Foundation models: forecasting with zero training

Everything so far learned *only* from the 5 series in front of us. **Time series foundation models**
flip this: they are pretrained on enormous corpora of real and synthetic time series, so they can
forecast a brand-new dataset **zero-shot** — no training required.

AG-TS ships two state-of-the-art families:

- **[Chronos-2](https://huggingface.co/autogluon/chronos-2)** (Amazon) — natively supports covariates
  and multivariate forecasting, up to 8192 context length.
- **[Toto-2](https://huggingface.co/collections/Datadog/toto-20)** (Datadog) — a decoder-only
  foundation model, available in sizes from 4M to 2.5B parameters.

The first `predict` call downloads the model weights (cached afterwards). After that, forecasting is
near-instant — and, as we'll see, more accurate than the models we trained above.

In [ ]:
fm_predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    target="target",
    eval_metric="MASE",
).fit(
    train_data,
    hyperparameters={
        "Chronos2": {},                          # Chronos-2 (120M), zero-shot
        "Toto2": {"model_path": "Toto-2.0-22m"}, # Toto-2 (22M), zero-shot
    },
    enable_ensemble=False,
)

Notice the `fit` call returns almost immediately: these models don't train on our data, they only run
inference. Let's see how zero-shot foundation models stack up against the ML models from Section 2.

In [ ]:
fm_predictor.leaderboard(test_data)

With **zero training on our data**, the foundation models match or beat the tuned ML ensemble. This is
the headline result: for many datasets, a pretrained model out of the box is a stronger starting point
than a bespoke pipeline.

Foundation models also combine well with everything else. The built-in presets do exactly this —
`"medium_quality"`, `"high_quality"`, and `"best_quality"` fit statistical, ML, and foundation models
and ensemble them together:

```python
predictor = TimeSeriesPredictor(prediction_length=48, target="target").fit(train_data, presets="best_quality")
```

## 4. Forecasting with covariates and fine-tuning

The previous sections used only the target's own history. Real forecasts often have side information —
**covariates** — that drives the target:

- **Known covariates** are known into the future: holidays, promotions, day-of-week, weather *forecasts*.
- **Past covariates** are only observed up to now: e.g. realized (measured) temperature.

To showcase this, we switch to an **electrical load forecasting** dataset (the BuildingsBench "Bull"
dataset): **hourly electricity load for 41 commercial buildings**, each shipped with local weather.
Building load is driven heavily by weather — hotter days mean more air conditioning — so air
temperature, dew-point temperature, and sea-level pressure make natural **known covariates** (in
production, these would come from a weather forecast). Chronos-2 supports them natively.

In [ ]:
data = TimeSeriesDataFrame.from_path("https://autogluon.s3.amazonaws.com/datasets/timeseries/bull/test.parquet", id_column="id")
data.head()

The columns are the target `load` plus three weather covariates. We forecast the next 24 hours and
build the known-covariate frame for the forecast horizon.

In [ ]:
prediction_length = 24
known_covariate_names = ["airtemperature", "dewtemperature", "sealvlpressure"]

train_data, test_data = data.train_test_split(prediction_length)

# Weather values over the forecast horizon (in practice, a weather forecast)
known_covariates = test_data.slice_by_timestep(-prediction_length, None).drop(columns=["load"])

### Zero-shot: with vs. without covariates

First, a **univariate** baseline — Chronos-2 using only the load history (weather columns dropped).

In [ ]:
predictor_univariate = TimeSeriesPredictor(
    prediction_length=prediction_length,
    target="load",
    eval_metric="MASE",
).fit(
    train_data[["load"]],
    hyperparameters={"Chronos2": {}},
)

Now the same model **with covariates** — the only change is declaring `known_covariates_names`. AG-TS
will condition Chronos-2's forecast on the weather.

In [ ]:
predictor_covariates = TimeSeriesPredictor(
    prediction_length=prediction_length,
    target="load",
    known_covariates_names=known_covariate_names,
    eval_metric="MASE",
).fit(
    train_data,
    hyperparameters={"Chronos2": {}},
)

Let's compare the two directly on the held-out test set.

In [ ]:
score_uni = predictor_univariate.evaluate(test_data[["load"]])["MASE"]
score_cov = predictor_covariates.evaluate(test_data)["MASE"]
print(f"Chronos-2 MASE without covariates: {-score_uni:.4f}")
print(f"Chronos-2 MASE with    covariates: {-score_cov:.4f}")

Adding covariates produces a more accurate forecast — same model, same data, just extra context. We can
quantify *which* covariates matter most with `feature_importance`, which measures the drop in accuracy
when each feature is shuffled.

In [ ]:
predictor_covariates.feature_importance(test_data, model="Chronos2", relative_scores=True)

Air temperature is by far the most useful signal for predicting building load — exactly what we'd
expect physically.

### Fine-tuning Chronos-2

Zero-shot is a great default, but when we have enough data we can squeeze out more accuracy by
**fine-tuning** the foundation model on our specific dataset. AG-TS makes this a one-line change: just
add `"fine_tune": True`. Here we fit both a zero-shot and a fine-tuned Chronos-2 (both using covariates)
so we can compare them directly.

In [ ]:
ft_predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    target="load",
    known_covariates_names=known_covariate_names,
    eval_metric="MASE",
).fit(
    train_data,
    hyperparameters={
        "Chronos2": [
            {"ag_args": {"name_suffix": "ZeroShot"}},
            {"fine_tune": True, "ag_args": {"name_suffix": "FineTuned"}},
        ],
    },
    time_limit=300,
    enable_ensemble=False,
)
ft_predictor.leaderboard(test_data)

Fine-tuning nudges accuracy up further. By default, Chronos-2 is fine-tuned with a lightweight LoRA
adapter; you can control the process via extra hyperparameters:

```python
hyperparameters={
    "Chronos2": {
        "fine_tune": True,
        "fine_tune_mode": "full",   # or "lora" (default)
        "fine_tune_lr": 1e-4,
        "fine_tune_steps": 2000,
    }
}
```

> **Tip:** Fine-tuning helps most when you have a reasonable number of series with sufficient history
> (rule of thumb: >100 series with median length > `3 * prediction_length`). With little data it can
> overfit. When unsure, the `chronos2_ensemble` preset combines zero-shot and fine-tuned Chronos-2 into
> a single robust predictor.

### Seeing the effect of covariates

Finally, let's *visualize* the difference. We forecast the same buildings with and without covariates
and plot them side by side. The covariate-aware forecast should track the true load's weather-driven
swings more closely.

In [ ]:
plot_item_ids = data.item_ids[:2].tolist()

# Univariate forecast (load history only)
pred_uni = predictor_univariate.predict(train_data[["load"]])
predictor_univariate.plot(test_data, pred_uni, item_ids=plot_item_ids, max_history_length=150)
plt.suptitle("Chronos-2 WITHOUT covariates")
plt.tight_layout()
plt.show()

# Covariate-aware forecast (uses known future weather)
pred_cov = predictor_covariates.predict(train_data, known_covariates=known_covariates)
predictor_covariates.plot(test_data, pred_cov, item_ids=plot_item_ids, max_history_length=150)
plt.suptitle("Chronos-2 WITH covariates")
plt.tight_layout()
plt.show()

> **Note:** More covariates don't always mean better forecasts. Always validate on held-out data —
> which AG-TS makes a one-liner.

## Summary & where to go next

We climbed the full forecasting ladder with one API:

1. **Baselines** set the bar (`SeasonalNaive` is a strong, cheap benchmark).
2. **ML models + ensembling** beat the baselines with a couple minutes of training.
3. **Foundation models** (Chronos-2, Toto-2) matched or beat the tuned pipeline **zero-shot**.
4. **Covariates** and **fine-tuning** sharpened the foundation model further on a real load-forecasting task.

Everything reduces to the same three lines when you want AutoGluon to make the choices:

```python
predictor = TimeSeriesPredictor(prediction_length=48, target="target").fit(train_data, presets="best_quality")
predictions = predictor.predict(train_data)
predictor.leaderboard(test_data)
```

**Learn more:**
- [AutoGluon Time Series tutorials](https://auto.gluon.ai/stable/tutorials/timeseries/index.html)
- [Forecasting with Chronos-2](https://auto.gluon.ai/stable/tutorials/timeseries/forecasting-chronos.html)
- [In-depth tutorial](https://auto.gluon.ai/stable/tutorials/timeseries/forecasting-indepth.html) — static features, missing values, backtesting, HPO
- [Evaluation metrics guide](https://auto.gluon.ai/stable/tutorials/timeseries/forecasting-metrics.html)
- [Chronos-2 technical report](https://arxiv.org/abs/2510.15821) · [fev-bench leaderboard](https://huggingface.co/spaces/autogluon/fev-bench)